In [ ]:
import random
import pandas as pd

In [17]:
# Load dataset
df = pd.read_csv('data/soc-sign-bitcoinotc.csv.gz', names=["SOURCE", "TARGET", "RATING", "TIME"])
df['TIME'] = pd.to_datetime(df['TIME'])

# Create neighbor mappings
outgoing_neighbors = df.groupby('SOURCE')['TARGET'].apply(set).to_dict()
incoming_neighbors = df.groupby('TARGET')['SOURCE'].apply(set).to_dict()

# Ensure all nodes are included, even if they are only targets
all_nodes = set(outgoing_neighbors.keys()).union(set(incoming_neighbors.keys()))

# Add missing nodes to outgoing_neighbors with no outgoing edges
for node in all_nodes:
    if node not in outgoing_neighbors:
        outgoing_neighbors[node] = set()

In [18]:
# Initialize the DLAS algorithm
def dlas(graph, num_iterations, sample_ratio):
    """
    DLAS algorithm for graph sampling.

    Parameters:
        graph (dict): The outgoing neighbors mapping (adjacency list).
        num_iterations (int): Number of iterations for the algorithm.
        sample_ratio (float): Percentage of nodes to sample from the graph.

    Returns:
        set: Sampled nodes.
    """
    nodes = list(graph.keys())
    num_nodes = len(nodes)
    sample_size = int(num_nodes * sample_ratio)
    
    # Initialize probability vectors for all nodes
    probabilities = {node: 1 / num_nodes for node in nodes}

    # Start from a randomly chosen node
    sampled_nodes = set()
    current_node = random.choice(nodes)

    for _ in range(num_iterations):
        # If we've sampled enough nodes, break the loop
        if len(sampled_nodes) >= sample_size:
            break

        # Choose the next action (neighbor node) based on probabilities
        neighbors = graph.get(current_node, [])
        
        # Exclude already sampled nodes from neighbors
        neighbors = [n for n in neighbors if n not in sampled_nodes]

        if not neighbors:
            # If no unvisited neighbors, pick a random unvisited node
            unvisited_nodes = [node for node in nodes if node not in sampled_nodes]
            if not unvisited_nodes:
                break  # All nodes have been sampled
            current_node = random.choice(unvisited_nodes)
            continue

        # Normalize probabilities for the neighbors
        probabilities_sum = sum(probabilities[n] for n in neighbors)
        normalized_probs = [probabilities[n] / probabilities_sum for n in neighbors]

        # Select the next node based on probabilities
        selected_node = random.choices(neighbors, weights=normalized_probs)[0]

        # Add the selected node to the sampled set
        sampled_nodes.add(selected_node)

        # Update the probabilities
        probabilities[selected_node] = 0  # Set probability to zero to avoid revisiting

        # Optionally, redistribute the probability mass to unvisited nodes
        total_prob = sum(probabilities.values())
        probabilities = {node: prob / total_prob for node, prob in probabilities.items()}

        # Move to the selected node
        current_node = selected_node

    return sampled_nodes

In [19]:
# Apply the DLAS algorithm with increased iterations
sampled_nodes = dlas(outgoing_neighbors, num_iterations=10000, sample_ratio=0.3)

# Output the sampled nodes
print(f"Orignal amount of nodes: {len(all_nodes)}")
print(f"Sampled Nodes ({len(sampled_nodes)} ({len(sampled_nodes)/len(all_nodes)}): {sampled_nodes}")


Orignal amount of nodes: 5881
Sampled Nodes (1764 (0.2999489882673015): {1, 2, 3, 4, 6, 7, 13, 15, 17, 19, 20, 21, 23, 25, 26, 29, 32, 33, 35, 36, 37, 39, 41, 45, 51, 53, 54, 57, 60, 61, 62, 64, 68, 69, 70, 72, 75, 76, 77, 78, 80, 81, 88, 93, 95, 96, 97, 101, 104, 108, 110, 111, 112, 113, 114, 115, 125, 129, 132, 135, 141, 142, 143, 144, 146, 147, 149, 152, 153, 159, 160, 163, 166, 167, 171, 177, 178, 179, 181, 183, 186, 192, 197, 198, 200, 202, 204, 206, 207, 209, 214, 215, 216, 219, 220, 221, 222, 228, 229, 230, 233, 235, 236, 238, 241, 242, 243, 245, 249, 256, 257, 261, 262, 265, 266, 268, 269, 270, 272, 273, 274, 276, 280, 282, 283, 284, 285, 291, 292, 295, 296, 297, 298, 300, 304, 309, 317, 318, 319, 320, 324, 325, 326, 330, 342, 346, 349, 350, 353, 357, 359, 361, 362, 363, 366, 372, 377, 378, 383, 384, 386, 387, 394, 395, 396, 399, 401, 402, 405, 408, 410, 412, 415, 416, 417, 418, 419, 423, 425, 427, 436, 438, 441, 443, 446, 452, 453, 454, 455, 459, 462, 463, 464, 465, 467, 468, 